In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

# Ensure project root is importable (so `src.*` imports work regardless of cwd)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model.svm_classifier import SVMClassifierModel

RANDOM_STATE = 42
N_SPLITS = 10

# Target encoding (as stated): 0=High, 1=Low, 2=Medium
CLASS_NAMES = ["High", "Low", "Medium"]
TARGET_COL = "risk_class_encoded"
META_COLS = ["risk_class", "risk_class_encoded", "latitude", "longitude"]

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "linear"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Feature columns = everything except metadata + target
FEATURE_COLS = [c for c in train_df.columns if c not in META_COLS]

# Define NUM_FEATURES / CAT_FEATURES based on dtypes in the processed dataset.
# (For this processed "linear" dataset, these are typically all numeric due to encoding + scaling.)
NUM_FEATURES = [c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(train_df[c])]
CAT_FEATURES = [c for c in FEATURE_COLS if c not in NUM_FEATURES]

X_train = train_df[FEATURE_COLS].to_numpy(dtype=np.float64)
y_train = train_df[TARGET_COL].to_numpy()

X_test = test_df[FEATURE_COLS].to_numpy(dtype=np.float64)
y_test = test_df[TARGET_COL].to_numpy()

print("=== Data Loaded ===")
print("train:", train_df.shape, "test:", test_df.shape)
print("#features:", len(FEATURE_COLS))
print("NUM_FEATURES:", len(NUM_FEATURES), "CAT_FEATURES:", len(CAT_FEATURES))
if CAT_FEATURES:
    print("CAT_FEATURES (non-numeric columns):", CAT_FEATURES)
print("y_train distribution:", dict(zip(*np.unique(y_train, return_counts=True))))

In [ ]:
def _decode_soil_group_for_g3(soil_group: pd.Series) -> pd.Series:
    """Return a numeric soil-group representation suitable for G3 formulas.

    Background
    ----------
    In some project artifacts, soil_group may be:
    - raw strings: 'A','B','C','D','Unknown'
    - ordinal encoded: Unknown=-1, A=0, B=1, C=2, D=3
    - robust-scaled ordinal (as in data/processed/linear): {-1, -0.5, 0, 0.5, 1}

    For the processed linear dataset in this repo, we observed the robust-scaled pattern
    and map it back to the ordinal scale so the G3 formulas match the original intent.
    """

    if soil_group.dtype == "O":
        mapping = {"A": 0.0, "B": 1.0, "C": 2.0, "D": 3.0, "Unknown": np.nan}
        return soil_group.map(mapping)

    soil_num = pd.to_numeric(soil_group, errors="coerce")

    # Detect the robust-scaled ordinal pattern.
    # In this repo’s processed linear data, soil_group takes values in {-1, -0.5, 0, 0.5, 1}.
    uniq = set(np.unique(np.round(soil_num.dropna().to_numpy(), 6)).tolist())
    scaled_pattern = {-1.0, -0.5, 0.0, 0.5, 1.0}
    if uniq and uniq.issubset(scaled_pattern):
        # Inverse of RobustScaler for this discrete ordinal feature.
        # (This matches a median=1, IQR=2 fit on the ordinal values.)
        return soil_num * 2.0 + 1.0

    # Otherwise: assume it is already on an ordinal-like numeric scale.
    return soil_num.astype(float)


def add_g3(df: pd.DataFrame) -> pd.DataFrame:
    """Add G3 domain interaction features.

    Definitions (from existing project notebooks):
    - Effective runoff:
        G3_effective_runoff = rainfall_intensity * (soil_group + 1) / 4
    - Soil-elevation risk:
        G3_soil_elev_risk = (soil_group + 1) / (elevation.clip(lower=0.5) + 1)

    Notes
    -----
    This experiment uses data/processed/linear/*.csv which is already scaled.
    These G3 features are therefore computed in the *processed feature space*.
    """

    out = df.copy()

    soil = _decode_soil_group_for_g3(out["soil_group"])  # numeric series

    rain = pd.to_numeric(out["historical_rainfall_intensity_mm_hr"], errors="coerce")
    elev = pd.to_numeric(out["elevation_m"], errors="coerce")

    out["G3_effective_runoff"] = (rain * (soil + 1.0)) / 4.0
    out["G3_soil_elev_risk"] = (soil + 1.0) / (elev.clip(lower=0.5) + 1.0)

    # Minimal safety: replace inf/-inf from any unexpected division with NaN then fill with 0.
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out[["G3_effective_runoff", "G3_soil_elev_risk"]] = out[["G3_effective_runoff", "G3_soil_elev_risk"]].fillna(0.0)

    return out


train_df_g3 = add_g3(train_df)
test_df_g3 = add_g3(test_df)

FEATURE_COLS_G3 = FEATURE_COLS + ["G3_effective_runoff", "G3_soil_elev_risk"]

X_train_g3 = train_df_g3[FEATURE_COLS_G3].to_numpy(dtype=np.float64)
X_test_g3 = test_df_g3[FEATURE_COLS_G3].to_numpy(dtype=np.float64)

print("=== G3 Features Added ===")
print("Added columns present in train:", all(c in train_df_g3.columns for c in ["G3_effective_runoff", "G3_soil_elev_risk"]))
print("X_train baseline:", X_train.shape, "X_train +G3:", X_train_g3.shape)

In [ ]:
sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)

print("=== Sample Weights ===")
print("min/mean/max:", float(sample_weight_train.min()), float(sample_weight_train.mean()), float(sample_weight_train.max()))
# Show effective total weight per class (should be ~balanced)
for cls in np.unique(y_train):
    w_sum = float(sample_weight_train[y_train == cls].sum())
    print(f"class={int(cls)} total_weight={w_sum:.2f} n={int((y_train==cls).sum())}")

In [ ]:
def _clone_custom_svm(model: SVMClassifierModel) -> SVMClassifierModel:
    """Safe clone for our custom estimator."""

    params = model.get_params(deep=True)
    return SVMClassifierModel(**params).build()


def _weighted_bootstrap_resample(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Resample (with replacement) using probabilities proportional to w."""

    w = np.asarray(w, dtype=np.float64)
    if w.ndim != 1 or w.shape[0] != y.shape[0]:
        raise ValueError("Weights must be 1D with same length as y")

    p = w / w.sum()
    idx = rng.choice(np.arange(y.shape[0]), size=y.shape[0], replace=True, p=p)
    return X[idx], y[idx]


def run_cv_and_test(
    model: SVMClassifierModel,
    X_train_in: np.ndarray,
    y_train_in: np.ndarray,
    X_test_in: np.ndarray,
    y_test_in: np.ndarray,
    sample_weight: np.ndarray | None = None,
    tag: str = "",
    class_names: list[str] | None = None,
    random_state: int = RANDOM_STATE,
    n_splits: int = N_SPLITS,
) -> dict:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_metrics = {
        "f2_macro": [],
        "precision_macro": [],
        "recall_macro": [],
        "f1_weighted": [],
    }

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_in, y_train_in), start=1):
        m = _clone_custom_svm(model)

        X_tr = X_train_in[tr_idx]
        y_tr = y_train_in[tr_idx]
        X_val = X_train_in[val_idx]
        y_val = y_train_in[val_idx]

        # Use sample weights via weighted bootstrap resampling (training only)
        if sample_weight is not None:
            rng = np.random.default_rng(random_state + fold)
            w_tr = sample_weight[tr_idx]
            X_fit, y_fit = _weighted_bootstrap_resample(X_tr, y_tr, w_tr, rng)
        else:
            X_fit, y_fit = X_tr, y_tr

        m.fit(X_fit, y_fit)
        y_pred = m.predict(X_val)

        fold_metrics["f2_macro"].append(
            fbeta_score(y_val, y_pred, beta=2, average="macro", zero_division=0)
        )
        fold_metrics["precision_macro"].append(
            precision_score(y_val, y_pred, average="macro", zero_division=0)
        )
        fold_metrics["recall_macro"].append(
            recall_score(y_val, y_pred, average="macro", zero_division=0)
        )
        fold_metrics["f1_weighted"].append(
            f1_score(y_val, y_pred, average="weighted", zero_division=0)
        )

        print(
            f"[{tag}] fold {fold:02d}/{n_splits} "
            f"f2={fold_metrics['f2_macro'][-1]:.4f} "
            f"prec={fold_metrics['precision_macro'][-1]:.4f} "
            f"rec={fold_metrics['recall_macro'][-1]:.4f} "
            f"f1w={fold_metrics['f1_weighted'][-1]:.4f}"
        )

    cv_mean = {k: float(np.mean(v)) for k, v in fold_metrics.items()}
    cv_std = {k: float(np.std(v)) for k, v in fold_metrics.items()}

    print("\n=== CV Summary" + (f" ({tag})" if tag else "") + " ===")
    for k in ["f2_macro", "precision_macro", "recall_macro", "f1_weighted"]:
        print(f"{k:16s}: {cv_mean[k]:.4f} ± {cv_std[k]:.4f}")

    # Final fit on full training set
    final_model = _clone_custom_svm(model)
    if sample_weight is not None:
        rng = np.random.default_rng(random_state + 999)
        X_fit, y_fit = _weighted_bootstrap_resample(X_train_in, y_train_in, sample_weight, rng)
    else:
        X_fit, y_fit = X_train_in, y_train_in

    final_model.fit(X_fit, y_fit)
    y_test_pred = final_model.predict(X_test_in)

    test_metrics = {
        "f2_macro": float(fbeta_score(y_test_in, y_test_pred, beta=2, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(y_test_in, y_test_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_test_in, y_test_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_test_in, y_test_pred, average="weighted", zero_division=0)),
    }

    print("\n=== Test Summary" + (f" ({tag})" if tag else "") + " ===")
    for k in ["f2_macro", "precision_macro", "recall_macro", "f1_weighted"]:
        print(f"{k:16s}: {test_metrics[k]:.4f}")

    labels = [0, 1, 2]
    target_names = class_names if class_names is not None else [str(x) for x in labels]
    print("\n=== Classification Report" + (f" ({tag})" if tag else "") + " ===")
    print(
        classification_report(
            y_test_in,
            y_test_pred,
            labels=labels,
            target_names=target_names,
            zero_division=0,
        )
    )

    return {
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "test": test_metrics,
        "y_test_pred": y_test_pred,
        "final_model": final_model,
    }

In [ ]:
svm_baseline = SVMClassifierModel(kernel="rbf").build()

results_baseline = run_cv_and_test(
    model=svm_baseline,
    X_train_in=X_train,
    y_train_in=y_train,
    X_test_in=X_test,
    y_test_in=y_test,
    sample_weight=sample_weight_train,
    tag="SVM_RBF_BASELINE",
    class_names=CLASS_NAMES,
)

In [ ]:
svm_g3 = SVMClassifierModel(kernel="poly", degree=3).build()

results_g3 = run_cv_and_test(
    model=svm_g3,
    X_train_in=X_train_g3,
    y_train_in=y_train,
    X_test_in=X_test_g3,
    y_test_in=y_test,
    sample_weight=sample_weight_train,
    tag="SVM_POLY3_PLUS_G3",
    class_names=CLASS_NAMES,
)

delta_cv_f2 = results_g3["cv_mean"]["f2_macro"] - results_baseline["cv_mean"]["f2_macro"]
delta_test_f2 = results_g3["test"]["f2_macro"] - results_baseline["test"]["f2_macro"]

print("\n=== +G3 Improvement Check ===")
print(f"Δ CV f2_macro   : {delta_cv_f2:+.4f}")
print(f"Δ Test f2_macro : {delta_test_f2:+.4f}")
if delta_cv_f2 > 0:
    print("✅ +G3 improved CV F2-macro")
else:
    print("⚠️  +G3 did not improve CV F2-macro")